# Analysis workflow tests

Loads the survey and sensor data, builds a panel dataset, and converts it for two kinds of analysis:

- [SurvivalScale](https://github.com/tazzben/Likert-estimation) (section 5), for Likert-scale responses as the dependent variable.
- OLS with [statsmodels](https://www.statsmodels.org/) (section 6), for a continuous dependent variable such as a slider response or a sensor measure.

Each section ends with assertions that fail loudly if something is wrong.

Install the estimators with `pip install SurvivalScale statsmodels` to run the final sections.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

for project_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = project_root / "src"
    if (src_dir / "CBACABLab").is_dir():
        sys.path.insert(0, str(src_dir))
        break

import CBACABLab

questionIdToQuestionName = {
    "QID1": "Q1",
    "QID2": "Q2",
    "QID3": "Q3",
    "QID4": "Q4",
    "QID7": "Q5",
    "QID5": "Q6",
    "QID6": "Q7"
}

DATABASE = "data.db"
SURVEY_FILE = "Survey.xlsx"
SENSOR_FOLDER = "./sensorData/"
REBUILD = True  # reload the survey and sensor data; set False to reuse the existing database

# Respondent-level slider and demographic questions
RESPONDENT_QUESTIONS = ["Q4_4", "Q5_4", "Q6", "Q7"]

# Questions with a Likert-style response, and the scale for each.
# This five-point scale matches the labels used by Q1-Q3 in Survey.xlsx.
AGREEMENT_SCALE = {
    "Strongly disagree": 0,
    "Somewhat disagree": 1,
    "Neither agree nor disagree": 2,
    "Somewhat agree": 3,
    "Strongly agree": 4,
}
LIKERT_SCALES = {question: AGREEMENT_SCALE for question in ["Q1", "Q2", "Q3"]}

TAGS = ["left", "right"]
INSTRUMENT_PROPERTIES = ["Attention"]

## 1. Load the survey and sensor data

In [2]:
if REBUILD:
    CBACABLab.LoadSurveyData(databaseName=DATABASE, surveyName=SURVEY_FILE)
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(0, 960, 0, 1080)],
        tag="left",
        rebuild=True,
        questionIdtoQuestionName=questionIdToQuestionName
    )
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(960, 1920, 0, 1080)],
        tag="right",
        questionIdtoQuestionName=questionIdToQuestionName
    )


In [3]:
# Test: the tables exist and the survey has data
with sqlite3.connect(DATABASE) as con:
    counts = {
        table: con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        for table in ["FullSurvey", "SensorData", "InstrumentData"]
    }
print(counts)
assert counts["FullSurvey"] > 0, "FullSurvey is empty"
if counts["SensorData"] == 0:
    print("WARNING: SensorData is empty. Sensor columns will be NaN; check the sensor folder and respondent ids.")

{'FullSurvey': 100, 'SensorData': 140, 'InstrumentData': 4993}


In [4]:
# Test: every question we plan to use is in the survey
available = set(CBACABLab.ListQuestions(DATABASE))
wanted = set(RESPONDENT_QUESTIONS) | set(LIKERT_SCALES)
missing = wanted - available
assert not missing, f"Questions not found in the survey: {sorted(missing)}"

## 2. Build the panel dataset

In [5]:
panel = CBACABLab.CreatePanelData(
    question_level_columns=list(LIKERT_SCALES),
    respondent_level_columns=RESPONDENT_QUESTIONS,
    tagFilter=TAGS,
    instrumentProperties=INSTRUMENT_PROPERTIES,
    databaseName=DATABASE,
)
panel

,respondent_id,question,response,tag,pupil,duration,property,value,Q4_4,Q5_4,...,pupil_right_Q6,pupil_right_Q7,Attention_left_Q4_4,Attention_left_Q5_4,Attention_left_Q6,Attention_left_Q7,Attention_right_Q4_4,Attention_right_Q5_4,Attention_right_Q6,Attention_right_Q7
0,1,Q1,Strongly disagree,left,2.628579,343.472949,Attention,98.137547,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
1,1,Q1,Strongly disagree,right,2.714873,357.075008,Attention,97.839279,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
2,1,Q2,Neither agree nor disagree,left,2.776471,358.868249,Attention,97.348034,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
3,1,Q2,Neither agree nor disagree,right,2.833336,1916.269888,Attention,98.041817,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
4,1,Q3,Strongly agree,left,2.817981,409.204696,NaN,NaN,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
5,1,Q3,Strongly agree,right,2.836787,694.727733,Attention,97.550504,65.198,22.066,...,2.442082,2.688176,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899
6,2,Q1,Strongly agree,left,2.693345,384.045313,Attention,97.755627,89.207,94.341,...,2.568309,2.609598,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269
7,2,Q1,Strongly agree,right,2.841391,722.940208,Attention,97.696284,89.207,94.341,...,2.568309,2.609598,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269
8,2,Q2,Neither agree nor disagree,left,2.739849,1027.143104,Attention,95.377090,89.207,94.341,...,2.568309,2.609598,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269
9,2,Q2,Neither agree nor disagree,right,2.741312,398.548022,Attention,97.866264,89.207,94.341,...,2.568309,2.609598,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269


In [8]:
# Tests: panel structure
required = {"respondent_id", "question", "response", *RESPONDENT_QUESTIONS}
assert required <= set(panel.columns), f"Missing columns: {required - set(panel.columns)}"
assert set(panel["question"]) == set(LIKERT_SCALES), "Panel contains unexpected questions"
sensor_columns = [c for c in panel.columns if c.startswith(("pupil_", "duration_"))]
if counts["SensorData"] > 0:
    assert sensor_columns and panel[sensor_columns].notna().any().any(), "No sensor data matched the survey questions"

# Strictly one row per respondent, question, tag
assert not panel.duplicated(["respondent_id", "question", "tag"]).any(), "More than one row per respondent and question"

## 3. Convert responses to `k` and `bound`

Responses outside a question's scale (for example "Don't know") can't be converted, so they are listed and dropped first.

In [26]:
def labels(scale):
    if isinstance(scale, str):
        scale = dict(CBACABLab.ListLikertScales(show_values=True))[scale]
    return {str(label).strip().lower() for label in scale}

in_scale = pd.Series(
    [str(r).strip().lower() in labels(LIKERT_SCALES[q]) for q, r in zip(panel["question"], panel["response"])],
    index=panel.index,
)
print("Dropping responses outside their scale:")
print(panel.loc[~in_scale, ["question", "response"]].value_counts())
panel_likert = CBACABLab.AddLikertColumns(panel[in_scale], LIKERT_SCALES)
panel_likert[panel_likert["tag"].eq("left")]

Dropping responses outside their scale:
Series([], Name: count, dtype: int64)


,respondent_id,question,response,tag,pupil,duration,property,value,Q4_4,Q5_4,...,Attention_left_Q4_4,Attention_left_Q5_4,Attention_left_Q6,Attention_left_Q7,Attention_right_Q4_4,Attention_right_Q5_4,Attention_right_Q6,Attention_right_Q7,k,bound
0,1,Q1,Strongly disagree,left,2.628579,343.472949,Attention,98.137547,65.198,22.066,...,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899,0,4
2,1,Q2,Neither agree nor disagree,left,2.776471,358.868249,Attention,97.348034,65.198,22.066,...,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899,2,4
4,1,Q3,Strongly agree,left,2.817981,409.204696,NaN,NaN,65.198,22.066,...,98.064354,97.363049,96.500681,96.817112,NaN,98.262032,96.216690,97.864899,4,4
6,2,Q1,Strongly agree,left,2.693345,384.045313,Attention,97.755627,89.207,94.341,...,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269,4,4
8,2,Q2,Neither agree nor disagree,left,2.739849,1027.143104,Attention,95.377090,89.207,94.341,...,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269,2,4
10,2,Q3,Somewhat disagree,left,2.752320,736.664201,Attention,97.902746,89.207,94.341,...,98.198494,97.656063,NaN,97.985451,98.113396,97.982811,NaN,98.525269,1,4
12,3,Q1,Neither agree nor disagree,left,2.719926,380.960150,Attention,98.039979,100.0,2.458,...,97.860268,NaN,96.727371,97.198349,NaN,NaN,97.966064,NaN,2,4
14,3,Q2,Strongly disagree,left,2.702798,591.957592,Attention,97.798824,100.0,2.458,...,97.860268,NaN,96.727371,97.198349,NaN,NaN,97.966064,NaN,0,4
16,3,Q3,Somewhat disagree,left,2.662162,733.998717,Attention,97.131732,100.0,2.458,...,97.860268,NaN,96.727371,97.198349,NaN,NaN,97.966064,NaN,1,4
18,4,Q1,Somewhat agree,left,2.624093,385.729328,Attention,97.832081,51.432,7.629,...,97.286724,97.194704,97.614433,97.306240,97.260977,97.442814,NaN,98.067665,3,4


## 4. Prepare the data for SurvivalScale

SurvivalScale expects one row per observation with `id`, `block`, `question`, numeric feature columns, `k` and `bound`. Here `block` is the respondent, so errors can be clustered by respondent.

In [25]:
base = panel_likert.reset_index(drop=True)

FEATURES = ["pupil"]
survival = pd.DataFrame({
    "id":  base["respondent_id"],
    "block": base["respondent_id"],
    "question": base["question"],
    "pupil": pd.to_numeric(base["pupil"], errors="coerce"),
    "k": base["k"].astype(int),
    "bound": base["bound"].astype(int),
}).dropna()
survival.head(10)


,id,block,question,pupil,k,bound
0,1,1,Q1,2.628579,0,4
1,1,1,Q1,2.714873,0,4
2,1,1,Q2,2.776471,2,4
3,1,1,Q2,2.833336,2,4
4,1,1,Q3,2.817981,4,4
5,1,1,Q3,2.836787,4,4
6,2,2,Q1,2.693345,4,4
7,2,2,Q1,2.841391,4,4
8,2,2,Q2,2.739849,2,4
9,2,2,Q2,2.741312,2,4


## 5. Run the estimator

In [21]:

%pip install SurvivalScale 


from SurvivalScale import get_results

if get_results is not None:
    cj, beta, metrics = get_results(
        survival,
        bootstrap_iterations=100,
        alpha=0.05,
        columns=FEATURES,
        block_id="block",
    )
    assert cj is not None and beta is not None and metrics is not None
    display(beta)

Note: you may need to restart the kernel to use updated packages.
Performing bootstrap analysis to estimate confidence intervals and p-values...


100%|██████████| 100/100 [00:03<00:00, 31.81it/s]


,beta,marginal,marginal_discrete,lower_bound,upper_bound,p_value
pupil,-0.404321,-0.076266,-0.099725,-0.560669,-0.251826,0.0


## 6. OLS with statsmodels

OLS needs a continuous outcome. Likert responses are ordinal, so here the continuous slider response `Q4_4` is the dependent variable, with the Likert response `k` and the other slider `Q5_4` as regressors. Question fixed effects control for the three survey items, and standard errors are clustered by respondent because each respondent appears once per question.

In [23]:
%pip install statsmodels
import statsmodels.formula.api as smf

ols_data = survival.copy()
ols_data["question"] = ols_data["question"].astype("category")

ols_model = smf.ols("pupil ~ C(question)", data=ols_data).fit(
    cov_type="cluster", cov_kwds={"groups": ols_data["block"]}
)
print(ols_model.summary())

Note: you may need to restart the kernel to use updated packages.
                            OLS Regression Results                            
Dep. Variable:                  pupil   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                 -0.025
Method:                 Least Squares   F-statistic:                    0.2851
Date:                Wed, 04 Nov 2026   Prob (F-statistic):              0.758
Time:                        15:56:56   Log-Likelihood:                 61.098
No. Observations:                  59   AIC:                            -116.2
Df Residuals:                      56   BIC:                            -110.0
Df Model:                           2                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------